In [1]:
# --- A simple dataset is included here ---
# You can replace this with any list of DNA strings.
# This dataset is a classic example used in bioinformatics courses.
# The hidden "planted" motif is 8 characters long and has variations.
DNA_SEQUENCES = [
    "GGCGTTCAGGCA",
    "AAGAATCAGTCA",
    "CAAGGAGTTCGC",
    "CACGTCAATCAC",
    "CAGGTATTAGGA"
]

# Set the length of the motif (k-mer) you want to find
K_MER_LENGTH = 5

def build_profile(motifs):
    """
    Builds a profile matrix (as probabilities) from a list of motif strings.
    Uses pseudocounts (Laplace's Rule of Succession) to avoid zero probabilities.
    """
    k = len(motifs[0])
    num_motifs = len(motifs)

    # Initialize profile with pseudocounts of 1
    profile = {
        'A': [1] * k,
        'C': [1] * k,
        'G': [1] * k,
        'T': [1] * k
    }

    # Count the occurrences of each nucleotide at each position
    for dna_string in motifs:
        for i, nucleotide in enumerate(dna_string):
            if nucleotide in profile:
                profile[nucleotide][i] += 1

    # Convert counts to probabilities
    # The total count at each position is num_motifs + 4 (due to the 4 pseudocounts)
    total_with_pseudocounts = num_motifs + 4
    for nucleotide in profile:
        for i in range(k):
            profile[nucleotide][i] /= total_with_pseudocounts

    return profile

def score_kmer(kmer, profile):
    """
    Calculates the probability of a k-mer given a profile matrix.
    """
    score = 1.0
    for i, nucleotide in enumerate(kmer):
        if nucleotide in profile:
            score *= profile[nucleotide][i]
        else:
            # Handle unexpected characters, though it shouldn't happen with this data
            score = 0.0
            break
    return score

def find_best_kmer(sequence, k, profile):
    """
    Finds the single most probable k-mer in a sequence, given a profile.
    """
    best_kmer = sequence[0:k]
    best_score = score_kmer(best_kmer, profile)

    # Iterate through all possible k-mers in the sequence
    for i in range(1, len(sequence) - k + 1):
        current_kmer = sequence[i:i+k]
        current_score = score_kmer(current_kmer, profile)

        if current_score > best_score:
            best_score = current_score
            best_kmer = current_kmer

    return best_kmer

def get_consensus(profile):
    """
    Generates the consensus string from a profile matrix.
    """
    k = len(profile['A'])
    consensus = ""
    for i in range(k):
        max_prob = -1
        best_nuc = ''
        for nucleotide in ['A', 'C', 'G', 'T']:
            if profile[nucleotide][i] > max_prob:
                max_prob = profile[nucleotide][i]
                best_nuc = nucleotide
        consensus += best_nuc
    return consensus

def calculate_score(motifs, consensus):
    """
    Calculates the total score of a set of motifs by summing the
    Hamming distance (mismatches) from the consensus string.
    A lower score is better.
    """
    score = 0
    for motif in motifs:
        for i in range(len(motif)):
            if motif[i] != consensus[i]:
                score += 1
    return score

def greedy_motif_search(dna_list, k):
    """
    Performs the Greedy Motif Search algorithm.

    1. Iterates through all k-mers in the *first* sequence to use as a "seed".
    2. For each seed, it builds a profile and finds the best matching k-mer
       in all *other* sequences, updating the profile as it goes.
    3. It keeps track of the set of motifs that produced the best (lowest) score.
    """
    if not dna_list:
        return [], 0, ""

    # Initialize with the first k-mers from each sequence as a default
    best_motifs = [seq[0:k] for seq in dna_list]
    profile = build_profile(best_motifs)
    consensus = get_consensus(profile)
    best_score = calculate_score(best_motifs, consensus)

    # Iterate through every possible k-mer in the *first* sequence as a seed
    first_sequence = dna_list[0]
    for i in range(len(first_sequence) - k + 1):
        seed_kmer = first_sequence[i:i+k]
        current_motifs = [seed_kmer]

        # Build the profile up one sequence at a time
        for j in range(1, len(dna_list)):
            # Profile is built from *all motifs found so far*
            profile = build_profile(current_motifs)

            # Find the best k-mer in the next sequence
            next_motif = find_best_kmer(dna_list[j], k, profile)
            current_motifs.append(next_motif)

        # After building the full set of motifs, score it
        final_profile = build_profile(current_motifs)
        final_consensus = get_consensus(final_profile)
        current_score = calculate_score(current_motifs, final_consensus)

        # The "Greedy" part: keep the best set found so far
        if current_score < best_score:
            best_score = current_score
            best_motifs = current_motifs

    # After checking all seeds, return the best set found
    final_profile = build_profile(best_motifs)
    final_consensus = get_consensus(final_profile)

    return best_motifs, best_score, final_consensus

# --- Main execution ---
if __name__ == "__main__":
    print(f"Finding a motif of length {K_MER_LENGTH} in {len(DNA_SEQUENCES)} sequences.\n")
    print("Input Sequences:")
    for seq in DNA_SEQUENCES:
        print(f"  {seq}")
    print("-" * 30)

    motifs, score, consensus = greedy_motif_search(DNA_SEQUENCES, K_MER_LENGTH)

    print(f"\n--- Results ---")
    print(f"Consensus String: {consensus}")
    print(f"Best Score (Total Mismatches): {score}")
    print("\nBest Motifs Found (one from each sequence):")
    for motif in motifs:
        print(f"  {motif}")

Finding a motif of length 5 in 5 sequences.

Input Sequences:
  GGCGTTCAGGCA
  AAGAATCAGTCA
  CAAGGAGTTCGC
  CACGTCAATCAC
  CAGGTATTAGGA
------------------------------

--- Results ---
Consensus String: ATCAG
Best Score (Total Mismatches): 5

Best Motifs Found (one from each sequence):
  TTCAG
  ATCAG
  AGGAG
  ATCAC
  ATTAG
